### Prepare full-dataset LightGBM training

#### We'll use:

#### Training: 2011–2015 (up to 7 April 2015)
#### Validation: 8 April 2015 – 24 April 2016
#### Features: 11 existing forecasting features
Target: sales

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

import lightgbm as lgb
import joblib
import gc

BASE = Path("../Data/processed/full_dataset")
FEATURE_DIR = BASE / "featured"

FEATURES = [
    "lag_1", "lag_7", "lag_28",
    "rolling_mean_7", "rolling_mean_28",
    "day_of_week", "month", "is_event",
    "snap_CA", "snap_TX", "snap_WI"
]

TARGET = "sales"

TRAIN_END = "2015-04-07"
VALID_START = "2015-04-08"

print("Full-scale modeling environment ready.")
print("Feature count:", len(FEATURES))

Full-scale modeling environment ready.
Feature count: 11


### Load training data efficiently

In [2]:
feature_files = sorted(
    FEATURE_DIR.glob("sales_chunk_*.parquet")
)

train_parts = []
valid_parts = []

columns = ["date", TARGET] + FEATURES

for i, file in enumerate(feature_files, 1):

    df = pd.read_parquet(
        file,
        columns=columns
    )

    df["date"] = pd.to_datetime(df["date"])

    train_parts.append(
        df[df["date"] <= TRAIN_END]
    )

    valid_parts.append(
        df[df["date"] >= VALID_START]
    )

    del df
    gc.collect()

    print(f"Processed {i}/{len(feature_files)} chunks")

train_df = pd.concat(train_parts, ignore_index=True)
valid_df = pd.concat(valid_parts, ignore_index=True)

del train_parts, valid_parts
gc.collect()

print("\nTraining shape:", train_df.shape)
print("Validation shape:", valid_df.shape)

Processed 1/31 chunks
Processed 2/31 chunks
Processed 3/31 chunks
Processed 4/31 chunks
Processed 5/31 chunks
Processed 6/31 chunks
Processed 7/31 chunks
Processed 8/31 chunks
Processed 9/31 chunks
Processed 10/31 chunks
Processed 11/31 chunks
Processed 12/31 chunks
Processed 13/31 chunks
Processed 14/31 chunks
Processed 15/31 chunks
Processed 16/31 chunks
Processed 17/31 chunks
Processed 18/31 chunks
Processed 19/31 chunks
Processed 20/31 chunks
Processed 21/31 chunks
Processed 22/31 chunks
Processed 23/31 chunks
Processed 24/31 chunks
Processed 25/31 chunks
Processed 26/31 chunks
Processed 27/31 chunks
Processed 28/31 chunks
Processed 29/31 chunks
Processed 30/31 chunks
Processed 31/31 chunks

Training shape: (46649700, 13)
Validation shape: (12531390, 13)


### Check RAM And Data quality 

In [3]:
import psutil

process = psutil.Process()

print("RAM usage:", psutil.virtual_memory().percent, "%")
print("Available RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2), "GB")

print("\nTraining memory:",
      round(train_df.memory_usage(deep=True).sum() / (1024**3), 2), "GB")

print("Validation memory:",
      round(valid_df.memory_usage(deep=True).sum() / (1024**3), 2), "GB")

print("\nMissing values:")
print(train_df[FEATURES + [TARGET]].isnull().sum())

print("\nTarget statistics:")
print(train_df[TARGET].describe())

RAM usage: 54.8 %
Available RAM: 7.09 GB

Training memory: 3.95 GB
Validation memory: 1.06 GB

Missing values:
lag_1               30490
lag_7              213430
lag_28             853720
rolling_mean_7     213430
rolling_mean_28    853720
day_of_week             0
month                   0
is_event                0
snap_CA                 0
snap_TX                 0
snap_WI                 0
sales                   0
dtype: int64

Target statistics:
count    4.664970e+07
mean     1.085791e+00
std      3.936370e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+00
max      7.630000e+02
Name: sales, dtype: float64


### Optimize memory 

In [4]:
# Reduce memory usage before model training

for col in FEATURES:
    if col in ["day_of_week", "month",
               "is_event", "snap_CA", "snap_TX", "snap_WI"]:
        train_df[col] = train_df[col].astype("int8")
        valid_df[col] = valid_df[col].astype("int8")
    else:
        train_df[col] = train_df[col].astype("float32")
        valid_df[col] = valid_df[col].astype("float32")

train_df[TARGET] = train_df[TARGET].astype("float32")
valid_df[TARGET] = valid_df[TARGET].astype("float32")

print("Training memory:",
      round(train_df.memory_usage(deep=True).sum() / 1024**3, 2), "GB")

print("Validation memory:",
      round(valid_df.memory_usage(deep=True).sum() / 1024**3, 2), "GB")

Training memory: 1.65 GB
Validation memory: 0.44 GB


## Create LightGBM datasets

In [5]:
train_data = lgb.Dataset(
    train_df[FEATURES],
    label=train_df[TARGET],
    free_raw_data=False
)

valid_data = lgb.Dataset(
    valid_df[FEATURES],
    label=valid_df[TARGET],
    reference=train_data,
    free_raw_data=False
)

print("LightGBM datasets created successfully!")

LightGBM datasets created successfully!


### Train LightGBM model

In [6]:
params = {
    "objective": "regression",
    "metric": ["mae", "rmse"],
    "learning_rate": 0.05,
    "num_leaves": 63,
    "max_depth": -1,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "max_bin": 127,
    "verbosity": -1,
    "num_threads": 4,
    "seed": 42
}

full_lgb_model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[valid_data],
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(50)
    ]
)

Training until validation scores don't improve for 50 rounds
[50]	valid_0's l1: 0.955634	valid_0's rmse: 2.03394
[100]	valid_0's l1: 0.934347	valid_0's rmse: 2.01154
[150]	valid_0's l1: 0.931691	valid_0's rmse: 2.00851
[200]	valid_0's l1: 0.931316	valid_0's rmse: 2.00786
[250]	valid_0's l1: 0.930863	valid_0's rmse: 2.00658
[300]	valid_0's l1: 0.930763	valid_0's rmse: 2.00622
Early stopping, best iteration is:
[286]	valid_0's l1: 0.930733	valid_0's rmse: 2.00612


### Calculate final evalution metrics 

In [7]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)
import numpy as np

# Generate validation predictions
y_pred = full_lgb_model.predict(
    valid_df[FEATURES],
    num_iteration=full_lgb_model.best_iteration
)

y_true = valid_df[TARGET].values

mae = mean_absolute_error(y_true, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_true, y_pred)
)

wape = (
    np.abs(y_true - y_pred).sum()
    / np.abs(y_true).sum()
) * 100

print("FULL DATASET LIGHTGBM RESULTS")
print("Best iteration:", full_lgb_model.best_iteration)
print("MAE:", round(mae, 5))
print("RMSE:", round(rmse, 5))
print("WAPE:", round(wape, 2), "%")

FULL DATASET LIGHTGBM RESULTS
Best iteration: 286
MAE: 0.93073
RMSE: 2.00612
WAPE: 71.66 %


### Compare sample vs full data LIGHTGBM

In [12]:
import joblib

lgb_model = joblib.load("../models/sample_lightgbm.pkl")

print("Sample LightGBM model loaded successfully!")

Sample LightGBM model loaded successfully!


In [13]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

# Use full dataset validation observations
X_compare = valid_df[FEATURES]
y_true = valid_df["sales"].values

# Predictions from sample-trained LightGBM
sample_predictions = lgb_model.predict(X_compare)

# Predictions from full-data LightGBM
full_predictions = full_lgb_model.predict(
    X_compare,
    num_iteration=full_lgb_model.best_iteration
)

def evaluate_model(y_true, predictions):
    return {
        "MAE": mean_absolute_error(y_true, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_true, predictions)),
        "WAPE (%)": (
            np.abs(y_true - predictions).sum()
            / np.abs(y_true).sum()
        ) * 100
    }

comparison = pd.DataFrame({
    "Sample LightGBM": evaluate_model(y_true, sample_predictions),
    "Full Dataset LightGBM": evaluate_model(y_true, full_predictions)
}).T

comparison

,MAE,RMSE,WAPE (%)
Sample LightGBM,0.939400,2.095643,72.330259
Full Dataset LightGBM,0.930733,2.006124,71.662936


#### This is a positive result. Our full-dataset LightGBM is performing better than the sample-trained model on the exact same validation observations.

## INVENTORY DEMAND STATISTICS

In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

FEATURE_DIR = Path("../Data/processed/full_dataset/featured")

inventory_stats = []

for file in sorted(FEATURE_DIR.glob("sales_chunk_*.parquet")):

    df = pd.read_parquet(
        file,
        columns=["id", "date", "sales"]
    )

    df = df[df["date"] <= TRAIN_END]

    stats = df.groupby("id")["sales"].agg(
        avg_daily_demand="mean",
        demand_std="std"
    ).reset_index()

    inventory_stats.append(stats)

inventory_df = pd.concat(inventory_stats, ignore_index=True)

inventory_df = inventory_df.groupby("id").agg(
    avg_daily_demand=("avg_daily_demand", "mean"),
    demand_std=("demand_std", "mean")
).reset_index()

print("Products:", len(inventory_df))
inventory_df.head()

Products: 30490


,id,avg_daily_demand,demand_std
0,FOODS_1_001_CA_1_evaluation,0.812418,1.310810
1,FOODS_1_001_CA_2_evaluation,1.209150,1.786709
2,FOODS_1_001_CA_3_evaluation,1.256209,2.307262
3,FOODS_1_001_CA_4_evaluation,0.375163,0.727241
4,FOODS_1_001_TX_1_evaluation,0.598693,1.221563


### Calculate safety stock and reorder point 

In [15]:
LEAD_TIME = 7
Z_SCORE = 1.645

inventory_df["safety_stock"] = (
    Z_SCORE * inventory_df["demand_std"] * np.sqrt(LEAD_TIME)
)

inventory_df["reorder_point"] = (
    inventory_df["avg_daily_demand"] * LEAD_TIME
    + inventory_df["safety_stock"]
)

inventory_df["safety_stock"] = np.ceil(
    inventory_df["safety_stock"]
).astype(int)

inventory_df["reorder_point"] = np.ceil(
    inventory_df["reorder_point"]
).astype(int)

inventory_df.head(10)

,id,avg_daily_demand,demand_std,safety_stock,reorder_point
0,FOODS_1_001_CA_1_evaluation,0.812418,1.310810,6,12
1,FOODS_1_001_CA_2_evaluation,1.209150,1.786709,8,17
2,FOODS_1_001_CA_3_evaluation,1.256209,2.307262,11,19
3,FOODS_1_001_CA_4_evaluation,0.375163,0.727241,4,6
4,FOODS_1_001_TX_1_evaluation,0.598693,1.221563,6,10
5,FOODS_1_001_TX_2_evaluation,0.607190,1.112660,5,10
6,FOODS_1_001_TX_3_evaluation,0.386275,0.792671,4,7
7,FOODS_1_001_WI_1_evaluation,0.603922,0.934050,5,9
8,FOODS_1_001_WI_2_evaluation,0.480392,0.953432,5,8
9,FOODS_1_001_WI_3_evaluation,0.324183,0.746128,4,6


### Save the inventory recommendations to a CSV file 

In [16]:
OUTPUT_DIR = Path("../Data/processed/full_dataset/inventory")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

inventory_df.to_parquet(
    OUTPUT_DIR / "inventory_recommendations.parquet",
    index=False
)

inventory_df.to_csv(
    OUTPUT_DIR / "inventory_recommendations.csv",
    index=False
)

print("Saved:", len(inventory_df), "product-store recommendations")

Saved: 30490 product-store recommendations


### Indentify top 20 priority products 

In [17]:
top_priority = inventory_df.sort_values(
    by="reorder_point",
    ascending=False
).head(20)

top_priority[
    ["id", "avg_daily_demand", "demand_std",
     "safety_stock", "reorder_point"]
]

,id,avg_daily_demand,demand_std,safety_stock,reorder_point
7022,FOODS_3_090_CA_3_evaluation,139.286275,116.293304,507,1482
11985,FOODS_3_586_TX_2_evaluation,103.016993,31.460249,137,859
7020,FOODS_3_090_CA_1_evaluation,70.669281,61.740224,269,764
7025,FOODS_3_090_TX_2_evaluation,65.271895,57.869776,252,709
7029,FOODS_3_090_WI_3_evaluation,63.452941,58.066205,253,697
11986,FOODS_3_586_TX_3_evaluation,79.784967,26.021803,114,672
7026,FOODS_3_090_TX_3_evaluation,60.620915,50.801825,222,646
7024,FOODS_3_090_TX_1_evaluation,49.961438,49.555438,216,566
11982,FOODS_3_586_CA_3_evaluation,69.511765,17.461951,76,563
8645,FOODS_3_252_TX_2_evaluation,60.097386,23.906381,105,525


### Categorize inventory priority 

In [18]:
inventory_df["priority"] = pd.cut(
    inventory_df["reorder_point"],
    bins=[-1, 10, 50, 150, float("inf")],
    labels=["Low", "Medium", "High", "Critical"]
)

print(inventory_df["priority"].value_counts())

inventory_df.head()

priority
Low         19905
Medium       9091
High         1280
Critical      214
Name: count, dtype: int64


,id,avg_daily_demand,demand_std,safety_stock,reorder_point,priority
0,FOODS_1_001_CA_1_evaluation,0.812418,1.310810,6,12,Medium
1,FOODS_1_001_CA_2_evaluation,1.209150,1.786709,8,17,Medium
2,FOODS_1_001_CA_3_evaluation,1.256209,2.307262,11,19,Medium
3,FOODS_1_001_CA_4_evaluation,0.375163,0.727241,4,6,Low
4,FOODS_1_001_TX_1_evaluation,0.598693,1.221563,6,10,Low


### Save the final inventory analysis  

In [19]:
inventory_df.to_parquet(
    OUTPUT_DIR / "final_inventory_analysis.parquet",
    index=False
)

inventory_df.to_csv(
    OUTPUT_DIR / "final_inventory_analysis.csv",
    index=False
)

print("Inventory optimization completed.")
print("Total combinations:", len(inventory_df))

Inventory optimization completed.
Total combinations: 30490


## Applying Deep Learning model (We'll build an LSTM forecasting model using PyTorch and compare its performance against LightGBM.)

In [2]:
import sys
print(sys.executable)

c:\Users\lenovo\AppData\Local\Programs\Python\Python314\python.exe


In [3]:
%pip install torch

  Using cached filelock-4.2.4-py3-none-any.whl.metadata (2.0 kB)
  Using cached setuptools-84.2.1-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/126.7 MB ? eta -:--:--
   - -------------------------------------- 3.4/126.7 MB 18.6 MB/s eta 0:00:07
   - -------------------------------------- 5.5/126.7 MB 13.9 MB/s eta 0:00:09
   -- ------------------------------------- 7.1/126.7 MB 11.8 MB/s eta 0:00:11
   -- ------------------------------------- 8.4/126.7 MB 10.0 MB/s eta 0:00:12
   --- ------------------------------------ 9.7/126.7 MB 9.5 MB/s eta 0:00:13
   --- ------------------------------------ 11.8/126.7 MB 9.3 MB/s eta 0:00:13
   ----


[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.15.0+cpu
False


In [3]:
import subprocess

result = subprocess.run(
    ["powershell", "-Command",
     "Get-CimInstance Win32_VideoController | Select-Object Name"],
    capture_output=True,
    text=True
)

print(result.stdout)


Name                 
----                 
Intel(R) UHD Graphics





#### our computer has Intel UHD Graphics, not an NVIDIA GPU. So NVIDIA CUDA isn't supported.

### Deep Learning would be done in google collab 